# অলীকবচন — merged architecture (teammate's judge/NLI/math lanes + honest validation + lexical/retrieval features)

**What this combines:**
- **From the 0.77 notebook (kept close to verbatim -- proven):** robust column/file auto-detection, claim-level SummaC-style NLI (catches one fabricated sentence buried in an otherwise-correct response, instead of averaging it away), ~15 deterministic math word-problem solvers, and a dual-prompt (Bengali + English, opposite polarity) Gemma-4-12B judge with the disagreement `|pA-pB|` as its own feature.
- **From the honest-validation pipeline:** nested-CV threshold selection (the 0.77 notebook tunes its threshold in-sample -- this fixes that), an unweighted stacked ensemble (no double imbalance correction), and lexical n-gram / entropy / novelty features that cost nothing and work independently of the judge.
- **New in this merge:** the judge's no-context ("closed-book") calls are RAG-augmented -- if an offline corpus is attached and retrieval finds a match for a no-context row, that evidence is injected into both judge prompts, upgrading a purely-parametric guess into a grounded one for exactly the rows where the other lanes (NLI, lexical overlap) have the least to work with.

**Requires**, same as the source notebook: an offline `transformers` wheelhouse (for Gemma-4 support), the `google/gemma-4/transformers/gemma-4-12b-it` Kaggle Model attached, and the mDeBERTa multilingual-NLI dataset attached. Optionally, set `CFG['corpus_path']` to an offline Bengali corpus JSON to enable the RAG augmentation -- everything degrades gracefully if it's absent.

## Setup — offline `transformers` upgrade (unchanged from source)

In [ ]:
# ---------- bootstrap: upgrade transformers from a local wheelhouse (NO internet needed/used) ----------
# gemma4_unified is only recognized by transformers>=5.13; Kaggle's frozen image ships 5.0.0.
# If no wheelhouse is attached (e.g. it's a teammate's private asset you don't have access to),
# this now falls back to Gemma-2-9B-IT below instead of crashing -- that model works fine with
# Kaggle's stock transformers, so no upgrade is required for it.
import subprocess, sys, glob, os

hits = glob.glob('/kaggle/input/**/wheelhouse', recursive=True)
WHEELHOUSE_AVAILABLE = bool(hits)

if WHEELHOUSE_AVAILABLE:
    WHEELDIR = hits[0]
    print('wheelhouse found at:', WHEELDIR, '|', len(glob.glob(f"{WHEELDIR}/*.whl")), 'wheels')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', WHEELDIR,
                       '-U', 'transformers', 'accelerate', 'bitsandbytes'],
                       capture_output=True, text=True, timeout=300)
    print(r.stdout[-2000:])
    if r.returncode != 0:
        print('STDERR:', r.stderr[-2000:])
    WHEELHOUSE_AVAILABLE = (r.returncode == 0)
    if not WHEELHOUSE_AVAILABLE:
        print('[bootstrap] wheelhouse install failed -- falling back to Gemma-2-9B-IT (see note below).')
else:
    print("[bootstrap] no offline wheelhouse attached under /kaggle/input -- skipping the transformers\n"
          "  upgrade. Gemma-4 needs transformers>=5.13; this notebook will use Gemma-2-9B-IT as the\n"
          "  judge instead, which works with Kaggle's stock transformers. To use Gemma-4 instead,\n"
          "  either get the wheelhouse dataset shared with you, or build your own (see chat).")

import transformers
print('transformers version:', transformers.__version__, '| wheelhouse upgrade applied:', WHEELHOUSE_AVAILABLE)


## Config + device (unchanged from source)

In [ ]:
import os
# fight fragmentation (must be set before the first CUDA allocation)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
import re, glob, json, random, gc, warnings
import numpy as np, pandas as pd
import torch
warnings.filterwarnings('ignore')

# No hard assert: if no local mDeBERTa dataset is attached, NLI_PATH stays None and Lane A
# (below) will try a network download, then degrade gracefully to neutral NLI features if
# that also fails -- rather than crashing the whole notebook here.
nli_hits = [p for p in glob.glob('/kaggle/input/**/config.json', recursive=True) if 'mdeberta' in p.lower()]
NLI_PATH = os.path.dirname(nli_hits[0]) if nli_hits else None
print('NLI model resolved at:', NLI_PATH if NLI_PATH else '(not found locally -- Lane A will try to fetch it)')

CFG = {
    # ---------- paths ----------
    'comp_dir'  : None,    # None -> auto-scan /kaggle/input/bengali-hallucination, /kaggle/input, ./data
    'test_csv'  : None,
    'labeled_csv': None,   # discovery also reads .json (comp ships 'dataset samples.json')
    'nli_path'  : NLI_PATH,
    # ---------- column overrides (None = auto-detect) ----------
    'col_id': None, 'col_prompt': None, 'col_response': None, 'col_context': None, 'col_label': None,
    # ---------- inference ----------
    'llm_batch' : 8,
    'llm_token_budget': 12288,
    'nli_batch' : 64,
    'win_list'  : [2, 3],
    'max_ctx_chars': 3000, 'max_prompt_chars': 1500, 'max_resp_chars': 2500,
    'max_len_llm'  : 2048,
    'seed': 42,
}

random.seed(CFG['seed']); np.random.seed(CFG['seed']); torch.manual_seed(CFG['seed'])
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'no GPU — enable an accelerator',
      '| GPUs:', torch.cuda.device_count() if DEVICE == 'cuda' else 0)

## Merged config additions

In [ ]:
# =====================================================================
# CELL: extra CFG for the merged additions (appended after teammate's CFG cell)
# =====================================================================
CFG.update({
    'corpus_path': None,          # set to an offline Bengali corpus JSON path to enable RAG for no-context rows
    'n_folds': 5,
    'n_repeats': 3,
    'threshold_n_repeats': 5,
    'threshold_metric': 'macro',  # 'macro' or 'hallu_f1' (F1 on label=0) -- confirm which the rules actually grade
})
print('merged CFG extras:', {k: CFG[k] for k in ['corpus_path', 'n_folds', 'n_repeats',
                                                  'threshold_n_repeats', 'threshold_metric']})


## File discovery + column auto-detection (unchanged from source)

In [ ]:
# ---------- file discovery (CSV + JSON) + column auto-detection ----------
CAND = {
    'id'      : ['id', 'ID', 'Id', 'idx', 'index', 'sample_id'],
    'prompt'  : ['prompt', 'question', 'query', 'instruction', 'input'],
    'response': ['response', 'answer', 'candidate', 'output', 'model_response',
                 'generated_response', 'candidate_response', 'hypothesis'],
    'context' : ['context', 'passage', 'source', 'source_passage', 'evidence',
                 'document', 'reference', 'source_text'],
    'label'   : ['label', 'target', 'TARGET', 'y', 'gold', 'is_faithful'],
}

def autodetect(cols, key):
    if CFG.get('col_' + key):
        return CFG['col_' + key]
    low = {c.lower(): c for c in cols}
    for cand in CAND[key]:
        if cand.lower() in low:
            return low[cand.lower()]
    for c in cols:
        if key in c.lower():
            return c
    return None

def resolve(df, keys):
    return {k: autodetect(list(df.columns), k) for k in keys}

def role_of(df):
    cols = list(df.columns)
    has = {k: autodetect(cols, k) is not None for k in CAND}
    if has['id'] and has['label'] and len(cols) <= 2:
        return 'sample_submission'
    if has['response'] and has['label']:
        return 'labeled'
    if has['response']:
        return 'test'
    return 'other'

def read_any(p):
    if p.lower().endswith('.csv'):
        return pd.read_csv(p)
    with open(p, encoding='utf-8') as f:
        obj = json.load(f)
    if isinstance(obj, list) and obj and isinstance(obj[0], dict):
        return pd.DataFrame(obj)
    raise ValueError('not a list-of-records json')

ROOTS = [r for r in ([CFG['comp_dir']] if CFG['comp_dir'] else
                     ['/kaggle/input/bengali-hallucination',
                      '/kaggle/input/competitions/bengali-hallucination',
                      '/kaggle/input', 'data']) if r and os.path.isdir(r)]
assert ROOTS, 'No input roots found — set CFG["comp_dir"].'

seen, files = set(), []
for root in ROOTS:
    for p in sorted(glob.glob(os.path.join(root, '**', '*.csv'), recursive=True)
                    + glob.glob(os.path.join(root, '**', '*.json'), recursive=True)):
        rp = os.path.realpath(p)
        if rp in seen or os.path.getsize(p) > 80_000_000:
            continue
        seen.add(rp); files.append(p)

test_df, labeled_df = None, None
for p in files:
    try:
        df = read_any(p)
    except Exception as e:
        if p.lower().endswith('.csv'):   # json misses are usually model configs — stay quiet
            print('skip', os.path.basename(p), '—', e)
        continue
    r = role_of(df)
    if r != 'other':
        print(f"{os.path.basename(p):40s} shape={str(df.shape):14s} -> {r}")
    if r == 'test'    and (test_df    is None or len(df) > len(test_df)):    test_df    = df.copy()
    if r == 'labeled' and (labeled_df is None or len(df) > len(labeled_df)): labeled_df = df.copy()

if CFG['test_csv']:    test_df    = read_any(CFG['test_csv'])
if CFG['labeled_csv']: labeled_df = read_any(CFG['labeled_csv'])
assert test_df is not None, 'Could not identify the test file — set CFG["test_csv"] explicitly.'

COLS  = resolve(test_df, ['id', 'prompt', 'response', 'context'])
LCOLS = resolve(labeled_df, ['id', 'prompt', 'response', 'context', 'label']) if labeled_df is not None else None
print('\nResolved test columns:', COLS)
if LCOLS: print('Resolved labeled columns:', LCOLS)
print('test:', test_df.shape, '| labeled sample:', None if labeled_df is None else labeled_df.shape)

## Data prep + `[NULL]`-marker normalization (unchanged from source)

In [ ]:
# ---------- prep ----------
NULLISH = {'', '[null]', 'null', 'none', 'nan', 'n/a'}   # the comp files use the literal string '[NULL]'

def clean(s, lim):
    if pd.isna(s):
        return ''
    s = str(s).strip()
    return '' if s.lower() in NULLISH else s[:lim]

def prep(df, cols):
    d = pd.DataFrame()
    d['id']       = df[cols['id']].values if cols['id'] else np.arange(len(df))
    d['prompt']   = df[cols['prompt']].map(lambda s: clean(s, CFG['max_prompt_chars'])).values if cols['prompt'] else ''
    d['response'] = df[cols['response']].map(lambda s: clean(s, CFG['max_resp_chars'])).values
    d['context']  = df[cols['context']].map(lambda s: clean(s, CFG['max_ctx_chars'])).values if cols['context'] else ''
    d['has_ctx']  = d['context'].str.len() > 0
    return d

test  = prep(test_df, COLS)
train = None
if labeled_df is not None and LCOLS and LCOLS['label']:
    train = prep(labeled_df, LCOLS)
    train['y'] = labeled_df[LCOLS['label']].astype(int).values   # 1 = faithful, 0 = hallucinated
    assert set(np.unique(train['y'])) <= {0, 1}, 'unexpected label values'
    print('sample label balance (1=faithful):')
    print(train['y'].value_counts(normalize=True).round(3).to_dict())
print('has_context — test:', round(test['has_ctx'].mean(), 3),
      '| sample:', None if train is None else round(train['has_ctx'].mean(), 3))

BN_SENT = re.compile(r'(?<=[।!?\n])\s+')   # Bengali danda-aware sentence split
def sents(t):
    parts = [s.strip() for s in BN_SENT.split(t) if s.strip()]
    return parts if parts else ([t] if t else [])

## Joggota — deterministic, zero-training-cost features
Style/entropy/length features, plus word-level novelty (not character-level -- Bengali's small alphabet makes char-level novelty nearly constant regardless of hallucination) and lexical n-gram overlap that works without any model, adapted to this notebook's `prep()` schema.

In [ ]:
# =====================================================================
# CELL: Joggota deterministic features -- cheap, zero-training-cost,
# complementary to the judge/NLI/math lanes. Adapted to this notebook's
# prep() schema (prompt / response / context / has_ctx).
# =====================================================================
import math
from collections import Counter

HEDGE_WORDS = ["সম্ভবত", "ধারণা করা হয়", "সাধারণত", "নির্দিষ্টভাবে"]
CULTURAL_WORDS = ["মুক্তিযুদ্ধ", "বঙ্গবন্ধু", "ভাষা আন্দোলন", "বাংলাদেশ"]
NUMBER_RE = re.compile(r"[0-9০-৯]+")


def _bn_words(text):
    return re.findall(r"[\u0980-\u09FF]+|\w+", str(text).lower())


def word_entropy(text):
    words = str(text).split()
    if len(words) < 2:
        return 0.0
    counts = Counter(words)
    total = len(words)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())


def char_entropy(text):
    text = str(text)
    if len(text) < 2:
        return 0.0
    counts = Counter(text)
    total = len(text)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())


def novel_word_ratio(context, response):
    """Only meaningful with real context -- comparing to the prompt instead
    would flag almost every legitimate answer as 'novel', since any real
    answer introduces info the question didn't contain. Returns 0 for
    no-context rows on purpose; the judge/NLI/retrieval lanes carry the
    grounding signal there instead."""
    if not str(context).strip():
        return 0.0
    ctx_words = set(_bn_words(context))
    resp_words = set(_bn_words(response))
    if not resp_words:
        return 0.0
    return len(resp_words - ctx_words) / len(resp_words)


def novel_number_ratio(context, response):
    resp_nums = set(NUMBER_RE.findall(str(response)))
    if not resp_nums or not str(context).strip():
        return 0.0
    ctx_nums = set(NUMBER_RE.findall(str(context)))
    return len(resp_nums - ctx_nums) / len(resp_nums)


def char_ngram_set(text, n=3):
    text = re.sub(r"\s+", " ", str(text)).strip()
    if len(text) < n:
        return {text} if text else set()
    return {text[i:i + n] for i in range(len(text) - n + 1)}


def char_ngram_similarity(a, b, n=3):
    sa, sb = char_ngram_set(a, n), char_ngram_set(b, n)
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def word_ngram_set(text, n=2):
    words = _bn_words(text)
    if len(words) < n:
        return set()
    return {tuple(words[i:i + n]) for i in range(len(words) - n + 1)}


def word_ngram_similarity(a, b, n=2):
    sa, sb = word_ngram_set(a, n), word_ngram_set(b, n)
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def hedge_density(text):
    words = str(text).split()
    n = max(len(words), 1)
    hits = sum(str(text).count(hw) for hw in HEDGE_WORDS)
    return hits / n


def contains_cultural_terms(text):
    text = str(text)
    return float(any(term in text for term in CULTURAL_WORDS))


def add_joggota_features(df):
    """In-place: adds style/lexical/grounding columns to a prep()'d dataframe."""
    premise = [c if c else p for c, p in zip(df['context'], df['prompt'])]
    df['length_ratio'] = (df['response'].str.len().clip(lower=1)
                           / df['prompt'].str.len().clip(lower=1))
    df['word_entropy'] = df['response'].apply(word_entropy)
    df['char_entropy'] = df['response'].apply(char_entropy)
    df['novel_word_ratio'] = [novel_word_ratio(c, r) for c, r in zip(df['context'], df['response'])]
    df['novel_number_ratio'] = [novel_number_ratio(c, r) for c, r in zip(df['context'], df['response'])]
    df['char_trigram_sim'] = [char_ngram_similarity(pr, r) for pr, r in zip(premise, df['response'])]
    df['word_bigram_sim'] = [word_ngram_similarity(pr, r) for pr, r in zip(premise, df['response'])]
    df['hedge_word_density'] = df['response'].apply(hedge_density)
    df['cultural_term_flag'] = df['prompt'].apply(contains_cultural_terms)
    return df


if train is not None:
    add_joggota_features(train)
add_joggota_features(test)
print('Joggota features added: length_ratio, word/char_entropy, novel_word/number_ratio, '
      'char_trigram_sim, word_bigram_sim, hedge_word_density, cultural_term_flag')


## Offline retrieval for no-context rows
Not a standalone feature pipeline -- its output feeds both a `corpus_match` feature *and* the judge prompts below, so a no-context row with a corpus hit gets a grounded judge call instead of a purely closed-book one.

In [ ]:
# =====================================================================
# CELL: offline retrieval for no-context rows (optional, degrades gracefully)
# Not a separate feature pipeline -- its output feeds BOTH a corpus_match
# feature AND the judge prompts below, so no-context rows get a grounded
# (RAG) judge call instead of a purely closed-book one whenever a corpus
# hit is found.
# =====================================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


class OfflineCorpusRetriever:
    def __init__(self, corpus_path):
        self.available = False
        if not corpus_path or not os.path.exists(corpus_path):
            print(f"[retrieval] no corpus attached (CFG['corpus_path']={corpus_path}) -- "
                  f"no-context rows fall back to the judge's closed-book framing only.")
            return
        with open(corpus_path, 'r', encoding='utf-8') as f:
            docs = json.load(f)
        self.texts = [d.get('text', '') for d in docs]
        if not self.texts:
            return
        self.vectorizer = TfidfVectorizer(max_features=50000, min_df=2)
        self.doc_matrix = self.vectorizer.fit_transform(self.texts)
        self.available = True
        print(f'[retrieval] loaded offline corpus: {len(self.texts)} documents')

    def best_match(self, query):
        if not self.available or not str(query).strip():
            return 0.0, ''
        q_vec = self.vectorizer.transform([str(query)])
        sims = cosine_similarity(q_vec, self.doc_matrix)[0]
        best_idx = int(np.argmax(sims))
        best_score = float(sims[best_idx])
        if best_score < 0.05:
            return 0.0, ''
        return best_score, self.texts[best_idx][:600]


def add_retrieval_features(df, retriever):
    """Only spends retrieval effort on no-context rows -- with-context rows
    already have real grounding for the NLI lane and the judge."""
    n = len(df)
    scores = np.zeros(n)
    evidence = [''] * n
    no_ctx_idx = np.where(~df['has_ctx'].values)[0]
    for i in no_ctx_idx:
        s, ev = retriever.best_match(df['prompt'].iat[i])
        scores[i] = s
        evidence[i] = ev
    df['corpus_match'] = scores
    return df, evidence


retriever = OfflineCorpusRetriever(CFG['corpus_path'])
train_evidence = None
if train is not None:
    train, train_evidence = add_retrieval_features(train, retriever)
test, test_evidence = add_retrieval_features(test, retriever)
print('[retrieval] no-context rows with a corpus hit -- test:',
      int(np.sum(test['corpus_match'] > 0)), '/', int((~test['has_ctx']).sum()))


## Lane A — claim-level NLI (unchanged from source)
SummaC-ZS style: response split into individual sentence-claims, context split into overlapping sentence windows across multiple window sizes, `min`-over-claims of max-entailment minus `max`-over-claims of max-contradiction. One fabricated sentence can't hide behind four correct ones.

In [ ]:
# ---------- Lane A: zero-shot multilingual NLI over context windows ----------
# Wrapped so a missing dataset AND no internet doesn't crash the notebook -- it degrades to
# neutral NLI features (X_of() already rescales NaN to a neutral 0.5) instead.
from transformers import AutoTokenizer, AutoModelForSequenceClassification

NLI_AVAILABLE = True
try:
    nli_id = CFG['nli_path'] or 'MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7'
    nli_tok = AutoTokenizer.from_pretrained(nli_id)
    try:
        nli_mod = AutoModelForSequenceClassification.from_pretrained(nli_id, dtype=torch.float16)
    except TypeError:
        nli_mod = AutoModelForSequenceClassification.from_pretrained(nli_id, torch_dtype=torch.float16)
    nli_mod = nli_mod.to(DEVICE).eval()
    lab2id = {v.lower(): k for k, v in nli_mod.config.id2label.items()}
    E_ID, C_ID = lab2id['entailment'], lab2id['contradiction']
    print('NLI labels:', nli_mod.config.id2label)
except Exception as e:
    NLI_AVAILABLE = False
    print(f"[NLI] could not load an mDeBERTa NLI model locally or over the network ({e}).\n"
          f"  Degrading gracefully: nli/nli_soft features will be neutral for every row.\n"
          f"  Attach the mDeBERTa dataset/model as a Kaggle input to enable this lane.")


@torch.no_grad()
def nli_batch(premises, hyps):
    out = []
    for i in range(0, len(premises), CFG['nli_batch']):
        enc = nli_tok(premises[i:i + CFG['nli_batch']], hyps[i:i + CFG['nli_batch']],
                      truncation=True, max_length=512, padding=True, return_tensors='pt').to(DEVICE)
        out.append(torch.softmax(nli_mod(**enc).logits.float(), -1).cpu().numpy())
    return np.concatenate(out)


def ctx_windows(ctx, win, cap=16):
    ss = sents(ctx)
    if len(ss) <= win:
        return [' '.join(ss)] if ss else []
    stride = max(1, win - 1)
    wins = [' '.join(ss[i:i + win]) for i in range(0, len(ss) - win + 1, stride)]
    wins.append(' '.join(ss[-win:]))
    return wins[:cap]


def nli_score_rows(df, tag=''):
    """SummaC-ZS style, ensembled over CFG['win_list'] window sizes.
    strict: min over claims of max-entailment  -  max over claims of max-contradiction
    soft  : mean over claims of (max-entailment - max-contradiction)     both in [-1, 1]"""
    strict = np.full(len(df), np.nan, dtype=float)
    soft = np.full(len(df), np.nan, dtype=float)
    if not NLI_AVAILABLE:
        return strict, soft  # all-neutral, no model to run
    pos = np.where(df['has_ctx'].values)[0]
    for j, i in enumerate(pos):
        ctx, resp = df['context'].iat[i], df['response'].iat[i]
        hyp = sents(resp)[:12] or [resp]
        st, so = [], []
        for win in CFG['win_list']:
            wins = ctx_windows(ctx, win) or [ctx]
            P, H = [], []
            for h in hyp:
                for w in wins:
                    P.append(w); H.append(h)
            probs = nli_batch(P, H).reshape(len(hyp), len(wins), -1)
            ent = probs[:, :, E_ID].max(axis=1)
            con = probs[:, :, C_ID].max(axis=1)
            st.append(float(ent.min() - con.max()))
            so.append(float((ent - con).mean()))
        strict[i] = float(np.mean(st)); soft[i] = float(np.mean(so))
        if (j + 1) % 200 == 0:
            print(f'  nli[{tag}] {j + 1}/{len(pos)}')
    return strict, soft


# run Lane A for both frames NOW, then give the GPU back before the judge loads
nli_tr = nli_score_rows(train, 'sample') if train is not None else None
nli_te = nli_score_rows(test, 'test')
if NLI_AVAILABLE:
    nli_mod = nli_mod.to('cpu'); gc.collect(); torch.cuda.empty_cache()
    print('NLI lane done -- model offloaded to CPU')
else:
    print('NLI lane skipped -- neutral features used instead')


## Deterministic math word-problem solver (unchanged from source)
~15 template solvers (work-rate, ratio, profit/loss, simple interest, LCM/GCD, combinatorics, averages...) that compute the real answer and compare it to the response's number within tolerance -- near-100%-precision, zero-training-cost labels for every row that matches a template.

In [ ]:
# ---------- deterministic math-word-problem solver (100% reproducible, no model) ----------
BN2EN = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
DAYS = ["রবিবার", "সোমবার", "মঙ্গলবার", "বুধবার", "বৃহস্পতিবার", "শুক্রবার", "শনিবার"]


def _nums(s):
    s = re.sub(r"(?<=[০-৯]),(?=[০-৯])", "", s)
    return [int(x.translate(BN2EN)) for x in re.findall(r"[০-৯]+", s)]


def _given_num(resp):
    resp = re.sub(r"(?<=[০-৯]),(?=[০-৯])", "", resp)
    m = re.findall(r"[০-৯]+", resp)
    return int(m[0].translate(BN2EN)) if m else None


def _close(a, b, tol=0.5):
    return a is not None and b is not None and abs(a - b) < tol


def solve_row(prompt, response):
    """Returns 1 (faithful) / 0 (hallucinated) / None (no template matched)."""
    n = _nums(prompt)
    g = _given_num(response)

    if "সপ্তাহের কোন দিন" in prompt or "সপ্তাহের কোন বার" in prompt:
        day_names = [d for d in DAYS if d in prompt]
        if not day_names or not n:
            return None
        idx = (DAYS.index(day_names[0]) + n[0]) % 7
        return int(DAYS[idx] in response)

    if len(n) < 2:
        return None

    if re.search("একা একটি কাজ|একাই.*দিনে সমাপ্ত", prompt):
        a, b = n[0], n[1]
        correct = (a * b) / (a + b) if (a + b) else 0
        return int(_close(correct, g))

    if "ক, খ ও গ" in prompt:
        a, b, c = n[0], n[1], n[2]
        rate = (1 / a if a else 0) + (1 / b if b else 0) + (1 / c if c else 0)
        correct = 1 / rate if rate else 0
        return int(_close(correct, g))

    if "বয়সের অনুপাত" in prompt:
        a, b, total = n[0], n[1], n[2]
        share = min(a, b)
        correct = total * share / (a + b)
        return int(_close(correct, g))

    if "সংখ্যার অনুপাত" in prompt:
        a, b, total = n[0], n[1], n[2]
        correct = total * b / (a + b)
        return int(_close(correct, g))

    if "ক্রয়মূল্য" in prompt or "কেনা হয়েছিল" in prompt:
        cost, pct = n[0], n[1]
        correct = cost * (1 - pct / 100) if "ক্ষতি" in prompt else cost * (1 + pct / 100)
        return int(_close(correct, g))

    if "প্রথম দফায়" in prompt or "প্রাথমিক মূল্য" in prompt:
        start_m = re.search(r"(?:প্রাথমিক মূল্য|শুরুর দাম)\s*([০-৯,]+)", prompt)
        if not start_m:
            return None
        start = int(re.sub(",", "", start_m.group(1)).translate(BN2EN))
        pcts = _nums(re.sub(r"(?:প্রাথমিক মূল্য|শুরুর দাম)\s*[০-৯,]+", "", prompt))
        p1, p2 = pcts[0], pcts[1]
        seg1 = prompt.split("এরপর" if "এরপর" in prompt else "পরে", 1)[0]
        seg2 = prompt.split("এরপর", 1)[-1] if "এরপর" in prompt else prompt.split("পরে", 1)[-1]
        s1 = 1 + p1 / 100 if ("বৃদ্ধি" in seg1 or "বেড়ে" in seg1) else 1 - p1 / 100
        s2 = 1 - p2 / 100 if ("কমে" in seg2 or "ছাড়" in seg2) else 1 + p2 / 100
        correct = start * s1 * s2
        return int(_close(correct, g))

    if "সরল সুদ" in prompt:
        principal, rate, time = n[0], n[1], n[2]
        correct = principal * rate * time / 100
        return int(_close(correct, g))

    if "অংশীদারের মধ্যে" in prompt:
        total, a, b, c = n[0], n[1], n[2], n[3]
        correct = total * b / (a + b + c)
        return int(_close(correct, g))

    if "একই দিকে দুইটি বাস" in prompt or "সাইকেল আরোহী একই বিন্দু" in prompt:
        s1, s2, t = n[0], n[1], n[2]
        correct = abs(s1 - s2) * t
        return int(_close(correct, g))

    if "দুইটি শহরের মধ্যে দূরত্ব" in prompt:
        dist, s1, s2 = n[0], n[1], n[2]
        correct = dist / (s1 + s2) if (s1 + s2) else 0
        return int(_close(correct, g))

    if "সংকেত বাতি" in prompt or "বাস স্টপেজ থেকে বাস" in prompt:
        from math import gcd
        a, b, c = n[0], n[1], n[2]
        lcm2 = a * b // gcd(a, b)
        correct = lcm2 * c // gcd(lcm2, c)
        return int(_close(correct, g))

    if "মিশ্রণে চিনি" in prompt:
        a, b, total = n[0], n[1], n[2]
        correct = total * b / (a + b)
        return int(_close(correct, g))

    if "প্যানেল গঠন" in prompt or "উপকমিটি গঠন" in prompt:
        from math import comb
        total, r = n[0], n[1]
        correct = comb(total, r)
        return int(_close(correct, g))

    if "রাশির গড়মান" in prompt or "শিক্ষার্থীর গড় নম্বর" in prompt:
        count, avg1, avg2 = n[0], n[1], n[2]
        correct = avg2 * (count + 1) - avg1 * count
        return int(_close(correct, g))

    return None


def math_feature(df):
    out = np.full(len(df), np.nan)
    for i in range(len(df)):
        lbl = solve_row(df['prompt'].iat[i], df['response'].iat[i])
        if lbl is not None:
            out[i] = lbl
    return out


math_tr = math_feature(train) if train is not None else None
math_te = math_feature(test)
print('math_solver matched -- sample:', None if math_tr is None else int(np.sum(~np.isnan(math_tr))), '/',
      0 if math_tr is None else len(math_tr),
      '| test:', int(np.sum(~np.isnan(math_te))), '/', len(math_te))

## Judge lane — Gemma-4-12B, dual-prompt, RAG-augmented for no-context rows
Loading, 4-bit quantization, and batching logic are the source notebook's (already proven). The one change: `prompt_A`/`prompt_B` now accept retrieved evidence, used only when a row has no real context, so retrieval-augmented no-context rows get a grounded judge call instead of a purely closed-book guess.

In [ ]:
# =====================================================================
# CELL: judge lane -- gemma-4-12b-it (or fallback), dual-prompt,
# RAG-augmented for no-context rows. Tries 3 candidate models in order;
# if none are accessible, skips the judge lane entirely with neutral
# (0.5) features rather than crashing the notebook.
# =====================================================================
import kagglehub
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

CANDIDATES = (['google/gemma-4/transformers/gemma-4-12b-it'] if WHEELHOUSE_AVAILABLE else []) + [
    'google/gemma-2/transformers/gemma-2-9b-it',
    'qwen-lm/qwen2.5/transformers/7b-instruct',
]
JUDGE_AVAILABLE, GEMMA_PATH = False, None
for handle in CANDIDATES:
    try:
        GEMMA_PATH = kagglehub.model_download(handle)
        JUDGE_AVAILABLE = True
        print('using judge model:', handle, '->', GEMMA_PATH)
        break
    except Exception as e:
        print(f'[judge] {handle} unavailable ({e}) -- trying next candidate...')

if JUDGE_AVAILABLE:
    tok = AutoTokenizer.from_pretrained(GEMMA_PATH)
    tok.padding_side = 'left'
    tok.truncation_side = 'left'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                              bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float32)
    kw = dict(device_map='auto', attn_implementation='eager', quantization_config=qcfg)
    try:
        llm = AutoModelForCausalLM.from_pretrained(GEMMA_PATH, dtype=torch.float16, **kw).eval()
    except TypeError:
        llm = AutoModelForCausalLM.from_pretrained(GEMMA_PATH, torch_dtype=torch.float16, **kw).eval()
    print('loaded judge model, 4-bit, eager attention')

    def token_set(variants):
        ids = []
        for v in variants:
            e = tok.encode(v, add_special_tokens=False)
            if len(e) == 1 and e[0] not in ids:
                ids.append(e[0])
        return ids

    YES = token_set(['Yes', 'yes', ' Yes', ' yes', 'হ্যাঁ'])
    NO = token_set(['No', 'no', ' No', ' no', 'না'])
    assert YES and NO and not set(YES) & set(NO)
else:
    print('[judge] no candidate model could be loaded (no access to any of them). '
          'Degrading gracefully: llm/pA/pB will be neutral (0.5) for every row.')


def chat(user_text):
    msgs = [{'role': 'user', 'content': user_text}]
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                       enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def prompt_A(p, r, c, ev=''):
    eff_ctx = c if c else ev
    ctx = f"\n\nতথ্যসূত্র (এটিই একমাত্র সত্যের ভিত্তি):\n{eff_ctx}" if eff_ctx else ""
    src = "প্রদত্ত তথ্যসূত্র" if eff_ctx else "বাস্তব, যাচাইযোগ্য জ্ঞান"
    return chat(f"""তুমি একজন কঠোর বাংলা ফ্যাক্ট-চেকার।{ctx}

প্রশ্ন/নির্দেশ:
{p}

মডেলের উত্তর:
{r}

উপরের উত্তরটি কি {src}-এর সাথে সম্পূর্ণ সঙ্গতিপূর্ণ — কোনো ভুল তথ্য, বানোয়াট নাম/তারিখ/সংখ্যা বা অসমর্থিত দাবি নেই? শুধু এক শব্দে উত্তর দাও: Yes অথবা No।""")


def prompt_B(p, r, c, ev=''):
    eff_ctx = c if c else ev
    ctx = f"\n\nSOURCE (the only ground truth):\n{eff_ctx}" if eff_ctx else ""
    tail = "not supported by the SOURCE" if eff_ctx else "false in the real world"
    return chat(f"""You are a strict fact-checker for Bengali LLM outputs.{ctx}

PROMPT:
{p}

RESPONSE:
{r}

Does the RESPONSE contain any hallucination — a factual error, a fabricated name/date/number, or a claim {tail}? Answer with exactly one word: Yes or No.""")


@torch.no_grad()
def judge(df, builder, flip, tag='', evidence=None):
    import time
    t_start = time.time()
    print(f'  judge[{tag}] starting -- {len(df)} rows to score')
    ev = evidence if evidence is not None else [''] * len(df)
    texts = [builder(df['prompt'].iat[i], df['response'].iat[i], df['context'].iat[i], ev[i])
             for i in range(len(df))]
    lens = np.array([len(x) for x in tok(texts, truncation=True,
                                         max_length=CFG['max_len_llm'])['input_ids']])
    order = np.argsort(lens, kind='stable')
    pair_ids = YES + NO
    ps, i, done, last_print = np.zeros(len(texts)), 0, 0, time.time()
    while i < len(order):
        take = int(min(CFG['llm_batch'], len(order) - i))
        while take > 1 and int(lens[order[i + take - 1]]) * take > CFG['llm_token_budget']:
            take -= 1
        idx = order[i:i + take]
        while True:
            try:
                enc = tok([texts[k] for k in idx], return_tensors='pt', padding=True,
                          truncation=True, max_length=CFG['max_len_llm']).to(llm.device)
                try:
                    out = llm(**enc, use_cache=False, logits_to_keep=1)
                except TypeError:
                    out = llm(**enc, use_cache=False)
                break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if len(idx) == 1:
                    raise
                idx = idx[:max(1, len(idx) // 2)]
                print(f'  judge[{tag}] OOM -> retry with batch {len(idx)}')
        logits = out.logits[:, -1, :].float()
        soft = torch.softmax(logits[:, pair_ids], -1).cpu().numpy()
        ps[idx] = soft[:, :len(YES)].sum(axis=1)
        i += len(idx); done += len(idx)
        # print every ~15s OR every 50 rows, whichever comes first -- never silent for long
        if time.time() - last_print > 15 or done % 50 < len(idx):
            elapsed = time.time() - t_start
            rate = done / max(elapsed, 1e-6)
            eta_min = (len(texts) - done) / max(rate, 1e-6) / 60
            print(f'  judge[{tag}] {done}/{len(texts)}  ({rate:.1f} rows/s, ~{eta_min:.1f} min left)')
            last_print = time.time()
    print(f'  judge[{tag}] done in {(time.time()-t_start)/60:.1f} min')
    return 1.0 - ps if flip else ps


if JUDGE_AVAILABLE:
    pA_tr = judge(train, prompt_A, flip=False, tag='sample/A', evidence=train_evidence) if train is not None else None
    pB_tr = judge(train, prompt_B, flip=True, tag='sample/B', evidence=train_evidence) if train is not None else None
    pA_te = judge(test, prompt_A, flip=False, tag='test/A', evidence=test_evidence)
    pB_te = judge(test, prompt_B, flip=True, tag='test/B', evidence=test_evidence)
    del llm, tok
    gc.collect(); torch.cuda.empty_cache()
    print('judge lane done')
else:
    pA_tr = np.full(len(train), 0.5) if train is not None else None
    pB_tr = np.full(len(train), 0.5) if train is not None else None
    pA_te = np.full(len(test), 0.5)
    pB_te = np.full(len(test), 0.5)
    print('judge lane skipped -- neutral (0.5) features used instead')


## Transfer-learning lane -- LaBSE + BanglaNLI-pretrained head (offline inference only)
The head was pretrained on `csebuetnlp/banglanli` in a **separate, internet-ON notebook** (train once, save `banglanli_head.pt`, publish as a Kaggle Dataset). This cell only **loads and runs** that pretrained head -- no training happens here, so it's fully offline-safe and there's no risk of collapse from training on the 299 competition rows.

**Set `CFG['pretrained_head_path']` below to your attached dataset's path.** If it's not attached, this degrades to a neutral feature and the rest of the notebook is unaffected.

*(See chat for the one-time pretraining script to run in a separate internet-ON notebook.)*

In [ ]:
# =====================================================================
# CELL: transfer-learning lane -- frozen LaBSE + a small head PRETRAINED
# on BanglaNLI (csebuetnlp/banglanli) in a separate internet-ON notebook,
# loaded here for offline inference only. No training happens in this
# (offline) kernel -- avoids the collapse risk of training anything on
# the 299 competition rows, and needs no internet at submission time.
# Degrades gracefully to a neutral feature if the pretrained head isn't
# attached yet.
# =====================================================================
!pip install -q sentence-transformers

CFG['pretrained_head_path'] = '/kaggle/input/YOUR-BANGLANLI-HEAD-DATASET/banglanli_head.pt'  # <-- SET THIS

from sentence_transformers import SentenceTransformer
import torch.nn as nn

TRANSFER_AVAILABLE = os.path.exists(CFG['pretrained_head_path'])
if TRANSFER_AVAILABLE:
    labse = SentenceTransformer('sentence-transformers/LaBSE', device=DEVICE)
    head = nn.Sequential(nn.Linear(3072, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 1)).to(DEVICE)
    head.load_state_dict(torch.load(CFG['pretrained_head_path'], map_location=DEVICE))
    head.eval()
    print('[transfer] loaded BanglaNLI-pretrained head from', CFG['pretrained_head_path'])

    def embed_pairs(prompts, contexts, responses, bs=128):
        premise = [c if c else p for c, p in zip(contexts, prompts)]
        pe = labse.encode(list(premise), batch_size=bs, convert_to_numpy=True, normalize_embeddings=True)
        re_ = labse.encode(list(responses), batch_size=bs, convert_to_numpy=True, normalize_embeddings=True)
        return np.concatenate([pe, re_, np.abs(pe - re_), pe * re_], axis=1)

    @torch.no_grad()
    def transfer_score(df):
        X = embed_pairs(df['prompt'], df['context'], df['response'])
        return torch.sigmoid(head(torch.tensor(X, dtype=torch.float32).to(DEVICE))).cpu().numpy().ravel()

    transfer_tr = transfer_score(train) if train is not None else None
    transfer_te = transfer_score(test)
    del labse, head; gc.collect(); torch.cuda.empty_cache()
    print('[transfer] lane done')
else:
    print(f"[transfer] no pretrained head found at {CFG['pretrained_head_path']} -- skipping, "
          f"neutral (0.5) feature used instead. Set CFG['pretrained_head_path'] above to enable.")
    transfer_tr = np.full(len(train), 0.5) if train is not None else None
    transfer_te = np.full(len(test), 0.5)


## Assemble the merged feature set (judge + NLI + math + lexical + retrieval)

In [ ]:
# =====================================================================
# CELL: assemble the merged feature set -- teammate's judge/NLI/math
# features plus this notebook's lexical/style/retrieval features.
# =====================================================================
def assemble(df, pA, pB, nli_pair, math_arr):
    s_nli, s_soft = nli_pair
    llm = (pA + pB) / 2
    return pd.DataFrame({
        'llm': llm, 'pA': pA, 'pB': pB,
        'nli': s_nli, 'nli_soft': s_soft,
        'has_ctx': df['has_ctx'].values,
        'resp_chars': df['response'].str.len().values,
        'ctx_chars': df['context'].str.len().values,
        'math_override': math_arr,
        'corpus_match': df['corpus_match'].values,
        'length_ratio': df['length_ratio'].values,
        'word_entropy': df['word_entropy'].values,
        'char_entropy': df['char_entropy'].values,
        'novel_word_ratio': df['novel_word_ratio'].values,
        'novel_number_ratio': df['novel_number_ratio'].values,
        'char_trigram_sim': df['char_trigram_sim'].values,
        'word_bigram_sim': df['word_bigram_sim'].values,
        'hedge_word_density': df['hedge_word_density'].values,
        'cultural_term_flag': df['cultural_term_flag'].values,
        'transfer_score': (transfer_tr if df is train else transfer_te),
    })


feat_tr = assemble(train, pA_tr, pB_tr, nli_tr, math_tr) if train is not None else None
feat_te = assemble(test, pA_te, pB_te, nli_te, math_te)
assert feat_tr is not None, 'labeled sample not found -- cannot fit the decision layer'


FEATURE_NAMES = [
    'llm', 'pA', 'pB', 'pA_pB_disagree', 'nli', 'nli_soft', 'has_ctx',
    'resp_chars', 'ctx_chars', 'math_override', 'corpus_match', 'length_ratio',
    'word_entropy', 'char_entropy', 'novel_word_ratio', 'novel_number_ratio',
    'char_trigram_sim', 'word_bigram_sim', 'hedge_word_density', 'cultural_term_flag',
    'transfer_score',
]


def X_of(feat, tag=''):
    cols = [
        feat['llm'].values, feat['pA'].values, feat['pB'].values,
        np.abs(feat['pA'].values - feat['pB'].values),
        (np.nan_to_num(feat['nli'].values, nan=0.0) + 1) / 2,
        (np.nan_to_num(feat['nli_soft'].values, nan=0.0) + 1) / 2,
        feat['has_ctx'].values.astype(float),
        np.minimum(feat['resp_chars'].values, CFG['max_resp_chars']) / CFG['max_resp_chars'],
        np.minimum(feat['ctx_chars'].values, CFG['max_ctx_chars']) / CFG['max_ctx_chars'],
        np.nan_to_num(feat['math_override'].values.astype(float), nan=0.5),
        feat['corpus_match'].values,
        np.minimum(feat['length_ratio'].values, 5.0) / 5.0,
        feat['word_entropy'].values / 6.0,
        feat['char_entropy'].values / 6.0,
        feat['novel_word_ratio'].values,
        feat['novel_number_ratio'].values,
        feat['char_trigram_sim'].values,
        feat['word_bigram_sim'].values,
        feat['hedge_word_density'].values,
        feat['cultural_term_flag'].values,
        feat['transfer_score'].values,
    ]
    X = np.column_stack(cols).astype(np.float64)

    # Diagnostic: report which feature(s) actually carried NaN/Inf before scrubbing them,
    # so a bad lane (most likely judge logits overflowing under 4-bit + eager attention)
    # is visible instead of silently masked.
    bad = ~np.isfinite(X)
    if bad.any():
        per_col = bad.sum(axis=0)
        offenders = {FEATURE_NAMES[i]: int(c) for i, c in enumerate(per_col) if c > 0}
        print(f"[X_of{'/' + tag if tag else ''}] non-finite values found and zero-filled: {offenders}")

    # Blanket safety net regardless of source -- LogisticRegression/GaussianNB can't
    # accept NaN, and StandardScaler will silently propagate NaN/Inf if not caught here.
    return np.nan_to_num(X, nan=0.5, posinf=1.0, neginf=0.0)


feat_tr.assign(y=train['y'].values).to_csv('sample_features.csv', index=False)
feat_te.to_csv('test_features.csv', index=False)
print('features assembled -- sample:', feat_tr.shape, '| test:', feat_te.shape,
      '| X width:', X_of(feat_tr).shape[1])


## Decision layer — stacked, unweighted, honestly validated
Replaces the source notebook's single in-sample GaussianNB with two diverse base models combined by a meta-learner, no class-weight double-correction, and a threshold chosen by nested cross-validation so the printed score is an honest estimate rather than one read off the same data it was tuned on.

In [ ]:
# =====================================================================
# CELL: decision layer -- stacked, unweighted, honestly-validated.
#
# What changed from a single in-sample GaussianNB:
#   - Two diverse, low-capacity base models (Logistic Regression, Gaussian
#     Naive Bayes) trained via repeated stratified CV for stable OOF
#     predictions, combined by a small Logistic Regression meta-learner
#     rather than trusting one model's calibration alone.
#   - No class_weight="balanced" anywhere -- with a tuned threshold doing
#     the imbalance correction already, adding weighted training on top
#     double-corrects and skews precision/recall in opposite directions
#     for the two classes.
#   - The threshold is chosen by NESTED cross-validation: picked on an
#     inner split, scored on a disjoint outer split, repeated. What's
#     printed is an honest estimate of leaderboard performance, not a
#     number read off the same data it was optimized on.
# =====================================================================
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report

BASE_MODEL_NAMES = ['logreg', 'nb']


def make_models(seed=CFG['seed']):
    return {
        'logreg': LogisticRegression(max_iter=2000, C=0.5, random_state=seed),
        'nb': GaussianNB(),
    }


def score_at_threshold(y_true, probs, t, metric):
    preds = (probs >= t).astype(int)
    if metric == 'hallu_f1':
        return f1_score(y_true, preds, pos_label=0)
    return f1_score(y_true, preds, average='macro')


def best_threshold(probs, labels, metric='macro'):
    grid = np.arange(0.05, 0.96, 0.01)
    best_t, best_f1 = 0.5, -1.0
    for t in grid:
        f1 = score_at_threshold(labels, probs, t, metric)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t), float(best_f1)


def honest_nested_threshold(probs, y, cfg=CFG, metric='macro'):
    rskf = RepeatedStratifiedKFold(n_splits=cfg['n_folds'], n_repeats=cfg['threshold_n_repeats'],
                                    random_state=cfg['seed'])
    fold_f1s, fold_thresholds = [], []
    for tr_idx, val_idx in rskf.split(probs.reshape(-1, 1), y):
        t, _ = best_threshold(probs[tr_idx], y[tr_idx], metric=metric)
        f1 = score_at_threshold(y[val_idx], probs[val_idx], t, metric)
        fold_f1s.append(f1)
        fold_thresholds.append(t)
    return float(np.median(fold_thresholds)), float(np.mean(fold_f1s)), float(np.std(fold_f1s))


def collect_base_oof(X, y, cfg=CFG):
    rskf = RepeatedStratifiedKFold(n_splits=cfg['n_folds'], n_repeats=cfg['n_repeats'],
                                    random_state=cfg['seed'])
    oof_sum = {name: np.zeros(len(y)) for name in BASE_MODEL_NAMES}
    oof_count = np.zeros(len(y))
    for tr_idx, val_idx in rskf.split(X, y):
        scaler = StandardScaler().fit(X[tr_idx])
        X_tr, X_val = scaler.transform(X[tr_idx]), scaler.transform(X[val_idx])
        y_tr = y[tr_idx]
        for name, m in make_models().items():
            m.fit(X_tr, y_tr)
            oof_sum[name][val_idx] += m.predict_proba(X_val)[:, 1]
        oof_count[val_idx] += 1
    oof_probs = {name: oof_sum[name] / np.maximum(oof_count, 1) for name in BASE_MODEL_NAMES}
    for name in BASE_MODEL_NAMES:
        f1 = f1_score(y, (oof_probs[name] >= 0.5).astype(int), average='macro')
        print(f"   base model '{name}': OOF macro-F1@0.5 = {f1:.4f}")
    return np.column_stack([oof_probs[n] for n in BASE_MODEL_NAMES])


def run_stacked_cv(X, y, cfg=CFG):
    oof_base_matrix = collect_base_oof(X, y, cfg)
    skf = StratifiedKFold(n_splits=cfg['n_folds'], shuffle=True, random_state=cfg['seed'])
    oof_meta = np.zeros(len(y))
    for tr_idx, val_idx in skf.split(oof_base_matrix, y):
        meta = LogisticRegression(max_iter=1000, C=1.0, random_state=cfg['seed'])
        meta.fit(oof_base_matrix[tr_idx], y[tr_idx])
        oof_meta[val_idx] = meta.predict_proba(oof_base_matrix[val_idx])[:, 1]

    print(f"\n[stacked] OOF macro-F1 @0.5 (uncalibrated) = "
          f"{f1_score(y, (oof_meta >= 0.5).astype(int), average='macro'):.4f}")

    t_macro, f1_macro_honest, f1_macro_std = honest_nested_threshold(oof_meta, y, cfg, metric='macro')
    t_hallu, f1_hallu_honest, f1_hallu_std = honest_nested_threshold(oof_meta, y, cfg, metric='hallu_f1')
    print(f"\n[stacked] HONEST nested-CV threshold estimate "
          f"({cfg['n_folds']}-fold x{cfg['threshold_n_repeats']} repeats):")
    print(f"   macro-F1: threshold≈{t_macro:.2f}  honest F1 = {f1_macro_honest:.4f} ± {f1_macro_std:.4f}")
    print(f"   hallu-F1: threshold≈{t_hallu:.2f}  honest F1 = {f1_hallu_honest:.4f} ± {f1_hallu_std:.4f}")

    print("\n[stacked] OOF classification report @ macro threshold:")
    print(classification_report(y, (oof_meta >= t_macro).astype(int), digits=3))

    chosen_thresh = t_macro if cfg['threshold_metric'] == 'macro' else t_hallu
    print(f"[stacked] using threshold_metric='{cfg['threshold_metric']}' -> final threshold={chosen_thresh:.2f}")
    return chosen_thresh, oof_base_matrix


def fit_full_stack_and_predict(X_train, y_train, X_test, oof_base_matrix, cfg=CFG):
    scaler = StandardScaler().fit(X_train)
    X_tr, X_te = scaler.transform(X_train), scaler.transform(X_test)
    base_test_probs = []
    for name, m in make_models(cfg['seed']).items():
        m.fit(X_tr, y_train)
        base_test_probs.append(m.predict_proba(X_te)[:, 1])
    test_base_matrix = np.column_stack(base_test_probs)

    meta = LogisticRegression(max_iter=1000, C=1.0, random_state=cfg['seed'])
    meta.fit(oof_base_matrix, y_train)
    print("\n[stacked] meta-learner coefficients (relative trust per base model):")
    for name, coef in zip(BASE_MODEL_NAMES, meta.coef_[0]):
        print(f'   {name:10s}: {coef:+.3f}')
    return meta.predict_proba(test_base_matrix)[:, 1]


y = train['y'].values
X, Xte = X_of(feat_tr, tag='train'), X_of(feat_te, tag='test')

print(f"[model] {CFG['n_folds']}-fold x{CFG['n_repeats']} repeated CV, "
      f"stacked LogReg + GaussianNB over {X.shape[1]} merged features:")
thresh, oof_base_matrix = run_stacked_cv(X, y, CFG)

print('\n[model] refitting base models + meta-learner on 100% of labeled data...')
test_probs = fit_full_stack_and_predict(X, y, Xte, oof_base_matrix, CFG)
preds = (test_probs >= thresh).astype(int)

sub = pd.DataFrame({'id': test['id'].values, 'label': preds})
sub.to_csv('submission.csv', index=False)
print('\nwrote submission.csv | balance (1=faithful):',
      sub['label'].value_counts(normalize=True).round(3).to_dict())
sub.head()
